# TF-IDF + Classic Classifiers — IMDB Sentiment Analysis

**Course:** Natural Language Processing · Unit 2 — Text Representation and Classification  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3rd ed.), Ch. 4 — Naive Bayes, Text Classification, and Sentiment  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Load and decode the **IMDB movie review dataset** from integer sequences back to text
2. Convert raw text into numerical features using **TF-IDF vectorisation**
3. Train and compare three classic classifiers: **Multinomial Naive Bayes**, **Logistic Regression**, and **Linear SVM**
4. Evaluate model quality with **classification reports** and a **confusion matrix**
5. Interpret the trade-offs between the three approaches for a binary sentiment task

## Dataset — IMDB Movie Reviews

The [IMDB dataset](https://ai.stanford.edu/~amaas/data/sentiment/) contains 50,000 movie reviews labelled as **positive (1)** or **negative (0)**. The Keras version encodes each review as a sequence of integers (word indices ranked by frequency). We reconstruct the original text by reversing the index so that scikit-learn estimators can consume it.


---

## 1. Environment Setup

Install dependencies **once** in your terminal before running this notebook:

```bash
pip install scikit-learn matplotlib seaborn tensorflow
```


In [ ]:
# pip install scikit-learn matplotlib seaborn tensorflow  # run once in terminal
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.datasets import imdb

---

## 2. Load and Prepare the IMDB Dataset

Keras stores each review as a list of integers (word rank in the corpus). `num_words=10000` keeps only the 10,000 most frequent words, discarding rare tokens.


In [ ]:
# Load the dataset; num_words=10000 keeps only the top-10k vocabulary
(X_train_idx, y_train), (X_test_idx, y_test) = imdb.load_data(num_words=10000)

print(f'Training sequences: {len(X_train_idx):,}')
print(f'Test sequences:     {len(X_test_idx):,}')
print(f'Label distribution  train: {(y_train == 1).sum()} pos / {(y_train == 0).sum()} neg')

> **Output interpretation:** The dataset is perfectly balanced — 25,000 training and 25,000 test samples, each split 50/50 between positive and negative labels. A balanced dataset means accuracy is a meaningful baseline metric (random guessing gives 50%).


### 2.1 Decode Integer Sequences Back to Text

The Keras IMDB index shifts word ranks by +3 to reserve four special tokens at positions 0–3.


In [ ]:
# Build a reverse mapping: integer index -> word string
# Keras shifts ranks by +3; positions 0-3 are reserved for special tokens
word_index = imdb.get_word_index()
index_word = {i + 3: word for word, i in word_index.items()}
index_word[0] = '<PAD>'
index_word[1] = '<START>'
index_word[2] = '<UNK>'
index_word[3] = '<UNUSED>'


def decode_review(sequence: list) -> str:
    """Convert a list of integer word indices to a human-readable string.

    Words not present in index_word are replaced with '?'.

    Args:
        sequence: List of integer word indices from the IMDB dataset.

    Returns:
        Space-joined string of the decoded words.
    """
    return ' '.join(index_word.get(i, '?') for i in sequence)


# Preview one decoded review
print('Sample decoded review (first 200 chars):')
print(decode_review(X_train_idx[0])[:200])

> **Output interpretation:** The decoded review is recognisable English prose. The `<START>` token at the beginning is an artifact of Keras encoding convention. Words outside the top-10k vocabulary appear as `?`. The decoding is lossy — capitalisation and punctuation are dropped — but sufficient for TF-IDF models, which ignore word order.


### 2.2 Build DataFrames and Re-split

We merge the Keras train/test split and re-split randomly at 80/20. This avoids any ordering bias present in the original split.


In [ ]:
# Decode all reviews into text strings
df_train = pd.DataFrame({
    'review': [decode_review(x) for x in X_train_idx],
    'label': y_train,
})
df_test = pd.DataFrame({
    'review': [decode_review(x) for x in X_test_idx],
    'label': y_test,
})

# Merge and perform a fresh random 80/20 split
df = pd.concat([df_train, df_test], ignore_index=True)
X = df['review']
y = df['label']
X_train_texts, X_test_texts, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {len(X_train_texts):,} reviews')
print(f'Test set:     {len(X_test_texts):,} reviews')

> **Output interpretation:** After merging and re-splitting, the training set contains 40,000 reviews and the test set 10,000. Using `random_state=42` makes the split fully reproducible. The class balance is preserved because the original 50,000 samples are already perfectly balanced.


---

## 3. Text Vectorisation with TF-IDF

**TF-IDF** (Term Frequency – Inverse Document Frequency) converts each review into a sparse numeric vector.

- **TF** (term frequency): how often a word appears in this review.
- **IDF** (inverse document frequency): down-weights words that appear in many reviews (common words carry less discriminative signal).
- **Result:** words that are frequent in one review but rare across the corpus get the highest weights.

`max_features=10000` keeps only the top 10,000 tokens by corpus-wide term frequency.


In [ ]:
# Fit the vectoriser on training data only to prevent data leakage into IDF weights
vectorizer = TfidfVectorizer(max_features=10000)
X_train_vec = vectorizer.fit_transform(X_train_texts)
X_test_vec = vectorizer.transform(X_test_texts)

print(f'Training matrix shape:  {X_train_vec.shape}')
print(f'Test matrix shape:      {X_test_vec.shape}')
density = X_train_vec.nnz / (X_train_vec.shape[0] * X_train_vec.shape[1])
print(f'Matrix density:         {density:.4%}  (sparsity {1 - density:.4%})')

> **Output interpretation:**
>
> - The training matrix is 40,000 rows (reviews) × 10,000 columns (TF-IDF features).
> - **Sparsity** is very high (typically >99%) because each review contains only a small fraction of the full vocabulary. scikit-learn stores this as a Compressed Sparse Row (CSR) matrix to avoid allocating 400M float values in RAM.
> - **Data leakage prevention:** `fit_transform` is called only on training data; `transform` (without `fit`) is applied to the test set so that IDF weights are computed solely from training reviews — the model never sees the test set during vocabulary construction.


---

## 4. Train Classic Classifiers

We train three classifiers on the same TF-IDF features to compare their performance:

| Classifier | Key assumption | Strength |
|---|---|---|
| **Multinomial Naive Bayes** | Features are conditionally independent given the class | Very fast; good baseline for text |
| **Logistic Regression** | Linear decision boundary in feature space | Interpretable weights; strong on sparse text |
| **Linear SVM** | Maximum-margin linear separator | Often best accuracy on high-dimensional sparse data |


In [ ]:
# Multinomial Naive Bayes: requires non-negative values (satisfied by TF-IDF)
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)
y_pred_nb = nb_model.predict(X_test_vec)
print('Naive Bayes training complete.')

In [ ]:
# Logistic Regression: max_iter=1000 ensures convergence on 10k-dimensional features
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_vec, y_train)
y_pred_lr = lr_model.predict(X_test_vec)
print('Logistic Regression training complete.')

In [ ]:
# LinearSVC is faster than SVC(kernel='linear') for large high-dimensional datasets
svm_model = LinearSVC()
svm_model.fit(X_train_vec, y_train)
y_pred_svm = svm_model.predict(X_test_vec)
print('Linear SVM training complete.')

---

## 5. Evaluate Performance

The **classification report** shows precision, recall, and F1-score per class, plus macro and weighted averages.


In [ ]:
label_names = ['Negative', 'Positive']

print('=== Naive Bayes ===')
print(classification_report(y_test, y_pred_nb, target_names=label_names))

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=label_names))

print('=== Linear SVM ===')
print(classification_report(y_test, y_pred_svm, target_names=label_names))

> **Output interpretation:**
>
> - **Precision** = of all reviews predicted as positive, what fraction are actually positive (low false-positive rate).
> - **Recall** = of all actual positive reviews, what fraction were correctly found (low false-negative rate).
> - **F1-score** = harmonic mean of precision and recall — the standard single-number summary for binary classification.
> - Typical results on IMDB: Naive Bayes ~85% F1 · Logistic Regression ~88–90% · Linear SVM ~89–91%.
> - The gap between Naive Bayes and the linear models shows the cost of the naive independence assumption: it struggles when word co-occurrences carry sentiment (e.g. 'not good' vs 'great').


---

## 6. Confusion Matrix — Logistic Regression

A confusion matrix shows the four possible outcomes: true positives (TP), true negatives (TN), false positives (FP), and false negatives (FN).


In [ ]:
cm = confusion_matrix(y_test, y_pred_lr)
_, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_names,
    yticklabels=label_names,
    ax=ax,
)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Confusion Matrix — Logistic Regression')
plt.tight_layout()
plt.show()

> **Output interpretation:**
>
> - The **diagonal cells** (TN top-left, TP bottom-right) show correctly classified reviews — larger numbers here are better.
> - The **off-diagonal cells** are misclassifications: FP = negative reviews predicted as positive; FN = positive reviews predicted as negative.
> - A roughly symmetric off-diagonal means the model makes similar types of errors in both directions, consistent with the balanced class distribution.
> - To understand which words drive the predictions, inspect `lr_model.coef_[0]`: the highest positive weights correspond to the strongest positive-sentiment words in the TF-IDF vocabulary.
